In [1]:
# Install if needed
# !pip install presidio-evaluator
# !pip install "presidio-analyzer>=2.2.360"
# !pip install gliner
# !python -m spacy download en_core_web_sm

In [2]:
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError
from presidio_evaluator.models import PresidioAnalyzerWrapper
from presidio_evaluator.experiment_tracking import get_experiment_tracker

from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import SpacyNlpEngine, NerModelConfiguration
from presidio_analyzer.predefined_recognizers.gliner_recognizer import GLiNERRecognizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2
%matplotlib inline

stanza and spacy_stanza are not installed
Flair is not installed by default


## 1. Load Dataset

In [3]:
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))

print(f"Loaded {len(dataset)} samples")

tokenizing input:   0%|          | 0/1500 [00:00<?, ?it/s]

loading model en_core_web_sm


tokenizing input:  74%|███████▍  | 1115/1500 [00:04<00:01, 225.91it/s]


KeyboardInterrupt: 

In [ ]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter

entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

## 2. Define GliNER Model Configurations

Popular GliNER models for PII detection:
-    "urchade/gliner_small-v2.1"
-    "nvidia/gliner-PII"
-    "gretelai/gretel-gliner-bi-large-v1.0"
-    "knowledgator/gliner-pii-base-v1.0"
-    "knowledgator/gliner-pii-large-v1.0"
-    "urchade/gliner_multi_pii-v1"

## Important: Entity Mapping

**Key Insight**: GliNER uses lowercase, space-separated entity labels (e.g., "person", "credit card"), while Presidio and the dataset use uppercase entity types (e.g., "PERSON", "CREDIT_CARD").

This notebook includes proper entity mapping (`GLINER_TO_PRESIDIO_MAPPING`) that ensures:
1. GliNER predictions are correctly mapped to Presidio entity types
2. Evaluation compares the right entities between predictions and ground truth
3. Results are accurate and meaningful

Without this mapping, evaluation would fail due to entity type mismatches!

In [ ]:
# Define GliNER models to test
GLINER_MODELS = [
    "urchade/gliner_small-v2.1",
    "nvidia/gliner-PII",
    "gretelai/gretel-gliner-bi-large-v1.0",
    "knowledgator/gliner-pii-base-v1.0",
    "knowledgator/gliner-pii-large-v1.0",
    "urchade/gliner_multi_pii-v1",
]

# Define entity label configurations (GliNER format: lowercase, space-separated)
ENTITY_CONFIGS = {
    "standard_pii": [
        "person", "organization", "location", "email", "phone number",
        "credit card number", "social security number", "date", "age", "address"
    ],
    "extended_pii": [
        "person", "organization", "location", "email", "phone number",
        "credit card number", "social security number", "date", "age", "address",
        "ip address", "url", "driver license", "passport", "bank account",
        "medical record number", "patient id", "insurance number"
    ],
    "minimal_pii": [
        "person", "email", "phone number", "credit card number", "ssn"
    ]
}

# Mapping from GliNER labels (lowercase) to Presidio types (uppercase)
GLINER_TO_PRESIDIO_MAPPING = {
    "person": "PERSON",
    "organization": "ORGANIZATION",
    "location": "LOCATION",
    "email": "EMAIL_ADDRESS",
    "phone number": "PHONE_NUMBER",
    "credit card": "CREDIT_CARD",
    "credit card number": "CREDIT_CARD",
    "social security number": "US_SSN",
    "ssn": "US_SSN",
    "date": "DATE_TIME",
    "time": "DATE_TIME",
    "age": "AGE",
    "address": "LOCATION",
    "ip address": "IP_ADDRESS",
    "url": "URL",
    "driver license": "US_DRIVER_LICENSE",
    "passport": "US_PASSPORT",
    "bank account": "US_BANK_NUMBER",
    "medical record number": "MEDICAL_LICENSE",
    "patient id": "ID",
    "insurance number": "ID",
}

# Threshold configurations
THRESHOLD_CONFIGS = [0.3, 0.4, 0.5]

## 3. Helper Functions for Creating Analyzer Engines

In [ ]:
def create_spacy_nlp_engine():
    """
    Create a SpaCy NLP engine using en_core_web_sm.
    This is used only for tokenization and linguistic features, NOT for NER.
    """
    model_config = NerModelConfiguration(
        labels_to_ignore=["O"],
        model_to_presidio_entity_mapping={},  # Empty - we don't use spaCy NER
    )
    
    nlp_engine = SpacyNlpEngine(
        models=[{"lang_code": "en", "model_name": "en_core_web_sm"}],
        ner_model_configuration=model_config
    )
    return nlp_engine

In [ ]:
def create_gliner_recognizer(model_name: str, gliner_labels: List[str], 
                            entity_mapping: Dict[str, str] = None,
                            threshold: float = 0.5, multi_label: bool = False):
    """
    Create a GliNER recognizer with specified model and labels.
    
    Args:
        model_name: HuggingFace model identifier
        gliner_labels: List of GliNER entity labels (lowercase, space-separated)
        entity_mapping: Mapping from GliNER labels to Presidio entity types
        threshold: Confidence threshold (default 0.5)
        multi_label: Whether to allow overlapping entities (default False)
    """
    if entity_mapping is None:
        entity_mapping = GLINER_TO_PRESIDIO_MAPPING
    
    # Get Presidio entity types from the mapping
    presidio_entities = list(set(entity_mapping.get(label, label.upper().replace(" ", "_")) 
                                 for label in gliner_labels))
    
    recognizer = GLiNERRecognizer(
        supported_entities=presidio_entities,  # Presidio format (uppercase)
        model_name=model_name,
        entity_mapping=gliner_labels,  # GliNER format (lowercase)
        model_to_presidio_mapping=entity_mapping,  # Conversion mapping
        threshold=threshold,
        multi_label=multi_label
    )
    return recognizer

In [ ]:
def create_registry_standalone_gliner(gliner_recognizer, nlp_engine):
    """
    Create a registry with ONLY GliNER recognizer (no other recognizers).
    SpacyRecognizer is explicitly removed.
    """
    registry = RecognizerRegistry()
    # Do NOT load predefined recognizers
    registry.add_recognizer(gliner_recognizer)
    return registry

def create_registry_gliner_with_others(gliner_recognizer, nlp_engine):
    """
    Create a registry with GliNER + other Presidio recognizers.
    SpacyRecognizer is explicitly removed to avoid spaCy NER predictions.
    """
    registry = RecognizerRegistry()
    registry.load_predefined_recognizers(nlp_engine=nlp_engine)
    
    # Remove SpacyRecognizer to avoid spaCy NER predictions
    registry.remove_recognizer("SpacyRecognizer")
    
    # Remove unnecessary recognizers
    unnecessary = [
        'NhsRecognizer', 'UkNinoRecognizer', 'SgFinRecognizer', 
        'AuAbnRecognizer', 'AuAcnRecognizer', 'AuTfnRecognizer', 
        'AuMedicareRecognizer', 'InPanRecognizer', 'InAadhaarRecognizer',
        'InVehicleRegistrationRecognizer', 'InPassportRecognizer', 
        'InVoterRecognizer'
    ]
    for rec in unnecessary:
        try:
            registry.remove_recognizer(rec)
        except:
            pass
    
    # Add GliNER recognizer
    registry.add_recognizer(gliner_recognizer)
    
    return registry

In [ ]:
def create_analyzer_engine(registry, nlp_engine, threshold: float = 0.3):
    """
    Create an AnalyzerEngine with the given registry and NLP engine.
    """
    analyzer = AnalyzerEngine(
        nlp_engine=nlp_engine,
        registry=registry,
        default_score_threshold=threshold
    )
    return analyzer

## 4. Align Dataset Entities

Map dataset entity types to Presidio entity types for evaluation.

In [ ]:
# Create entity mapping
entities_mapping = PresidioAnalyzerWrapper.presidio_entities_map.copy()
entities_mapping["TITLE"] = "TITLE"
entities_mapping["PREFIX"] = "TITLE"
entities_mapping["ZIP_CODE"] = "LOCATION"
entities_mapping["NRP"] = "ORGANIZATION"

print("Entities mapping:")
pprint(entities_mapping)

# Align dataset
dataset_aligned = SpanEvaluator.align_entity_types(
    dataset, 
    entities_mapping=entities_mapping, 
    allow_missing_mappings=True
)

new_entity_counts = get_entity_counts(dataset_aligned)
print("\nCount per entity after alignment:")
pprint(new_entity_counts.most_common(), compact=True)

dataset_entities = list(new_entity_counts.keys())

## 5. Experiment 1: Standalone GliNER (No Other Recognizers)

Test GliNER models alone without any pattern-based or other recognizers.

In [ ]:
# Results storage
standalone_results = {}

In [ ]:
%%time

print("=" * 80)
print("EXPERIMENT 1: STANDALONE GLINER")
print("=" * 80)

# Create NLP engine once
nlp_engine = create_spacy_nlp_engine()

# Test first model with standard PII entities
model_name = GLINER_MODELS[0]
entity_config = "standard_pii"
threshold = 0.4
multi_label = False

print(f"\n{'='*60}")
print(f"Model: {model_name}")
print(f"Entities: {entity_config}")
print(f"Threshold: {threshold}")
print(f"Multi-label: {multi_label}")
print(f"{'='*60}\n")

# Create GliNER recognizer
gliner_recognizer = create_gliner_recognizer(
    model_name=model_name,
    gliner_labels=ENTITY_CONFIGS[entity_config],
    entity_mapping=GLINER_TO_PRESIDIO_MAPPING,
    threshold=threshold,
    multi_label=multi_label
)

# Create registry with ONLY GliNER
registry = create_registry_standalone_gliner(gliner_recognizer, nlp_engine)

# Create analyzer
analyzer = create_analyzer_engine(registry, nlp_engine, threshold=threshold)

print("Loaded recognizers:")
pprint([rec.name for rec in analyzer.registry.get_recognizers("en", all_fields=True)])
print(f"\nSupported entities: {analyzer.get_supported_entities('en')}")

# Set up evaluator
experiment = get_experiment_tracker()
evaluator = SpanEvaluator(model=analyzer, iou_threshold=0.7)

# Track parameters
params = {
    "dataset_name": dataset_name,
    "model_name": evaluator.model.name,
    "gliner_model": model_name,
    "entity_config": entity_config,
    "threshold": threshold,
    "multi_label": multi_label,
    "mode": "standalone_gliner",
    "spacy_recognizer": "disabled"
}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset_aligned)

# Run evaluation
print("\nRunning evaluation...")
evaluation_results = evaluator.evaluate_all(dataset_aligned)
results = evaluator.calculate_score(evaluation_results)

# Store results
key = f"{model_name}_{entity_config}_{threshold}_standalone"
standalone_results[key] = results

# Log metrics
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
experiment.end()

# Print results
print("\n" + "=" * 60)
print("RESULTS:")
print(f"PII Precision: {results.pii_precision:.3f}")
print(f"PII Recall: {results.pii_recall:.3f}")
print(f"PII F-score: {results.pii_f:.3f}")
print("=" * 60)

## 6. Experiment 2: GliNER with Other Recognizers

Test GliNER models combined with pattern-based recognizers (SpacyRecognizer still disabled).

In [ ]:
# Results storage
combined_results = {}

In [ ]:
%%time

print("=" * 80)
print("EXPERIMENT 2: GLINER + OTHER RECOGNIZERS (SpacyRecognizer DISABLED)")
print("=" * 80)

# Create NLP engine once
nlp_engine = create_spacy_nlp_engine()

# Test first model with standard PII entities
model_name = GLINER_MODELS[0]
entity_config = "standard_pii"
threshold = 0.4
multi_label = False

print(f"\n{'='*60}")
print(f"Model: {model_name}")
print(f"Entities: {entity_config}")
print(f"Threshold: {threshold}")
print(f"Multi-label: {multi_label}")
print(f"{'='*60}\n")

# Create GliNER recognizer
gliner_recognizer = create_gliner_recognizer(
    model_name=model_name,
    gliner_labels=ENTITY_CONFIGS[entity_config],
    entity_mapping=GLINER_TO_PRESIDIO_MAPPING,
    threshold=threshold,
    multi_label=multi_label
)

# Create registry with GliNER + others (but NO SpacyRecognizer)
registry = create_registry_gliner_with_others(gliner_recognizer, nlp_engine)

# Create analyzer
analyzer = create_analyzer_engine(registry, nlp_engine, threshold=threshold)

print("Loaded recognizers:")
pprint([rec.name for rec in analyzer.registry.get_recognizers("en", all_fields=True)])
print(f"\nSupported entities: {analyzer.get_supported_entities('en')}")

# Verify SpacyRecognizer is not present
recognizer_names = [rec.name for rec in analyzer.registry.get_recognizers("en", all_fields=True)]
assert "SpacyRecognizer" not in recognizer_names, "SpacyRecognizer should be removed!"
print("\n✓ Confirmed: SpacyRecognizer is disabled")

# Set up evaluator
experiment = get_experiment_tracker()
evaluator = SpanEvaluator(model=analyzer, iou_threshold=0.7)

# Track parameters
params = {
    "dataset_name": dataset_name,
    "model_name": evaluator.model.name,
    "gliner_model": model_name,
    "entity_config": entity_config,
    "threshold": threshold,
    "multi_label": multi_label,
    "mode": "gliner_with_others",
    "spacy_recognizer": "disabled"
}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset_aligned)

# Run evaluation
print("\nRunning evaluation...")
evaluation_results = evaluator.evaluate_all(dataset_aligned)
results = evaluator.calculate_score(evaluation_results)

# Store results
key = f"{model_name}_{entity_config}_{threshold}_combined"
combined_results[key] = results

# Log metrics
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
experiment.end()

# Print results
print("\n" + "=" * 60)
print("RESULTS:")
print(f"PII Precision: {results.pii_precision:.3f}")
print(f"PII Recall: {results.pii_recall:.3f}")
print(f"PII F-score: {results.pii_f:.3f}")
print("=" * 60)

## 7. Experiment 3: Compare Different GliNER Models

Loop through different GliNER base models to compare performance.

In [ ]:
# Results storage
model_comparison_results = {}

In [ ]:
%%time

print("=" * 80)
print("EXPERIMENT 3: COMPARE DIFFERENT GLINER MODELS")
print("=" * 80)

# Create NLP engine once
nlp_engine = create_spacy_nlp_engine()

# Fixed configuration
entity_config = "standard_pii"
threshold = 0.4
multi_label = False
mode = "standalone"  # or "combined"

for model_name in GLINER_MODELS:
    print(f"\n{'='*60}")
    print(f"Testing Model: {model_name}")
    print(f"{'='*60}\n")
    
    try:
        # Create GliNER recognizer
        gliner_recognizer = create_gliner_recognizer(
            model_name=model_name,
            gliner_labels=ENTITY_CONFIGS[entity_config],
            entity_mapping=GLINER_TO_PRESIDIO_MAPPING,
            threshold=threshold,
            multi_label=multi_label
        )
        
        # Create registry
        if mode == "standalone":
            registry = create_registry_standalone_gliner(gliner_recognizer, nlp_engine)
        else:
            registry = create_registry_gliner_with_others(gliner_recognizer, nlp_engine)
        
        # Create analyzer
        analyzer = create_analyzer_engine(registry, nlp_engine, threshold=threshold)
        
        # Set up evaluator
        experiment = get_experiment_tracker()
        evaluator = SpanEvaluator(model=analyzer, iou_threshold=0.7)
        
        # Track parameters
        params = {
            "dataset_name": dataset_name,
            "model_name": evaluator.model.name,
            "gliner_model": model_name,
            "entity_config": entity_config,
            "threshold": threshold,
            "multi_label": multi_label,
            "mode": mode,
            "spacy_recognizer": "disabled"
        }
        params.update(evaluator.model.to_log())
        experiment.log_parameters(params)
        experiment.log_dataset_hash(dataset_aligned)
        
        # Run evaluation
        print("Running evaluation...")
        evaluation_results = evaluator.evaluate_all(dataset_aligned)
        results = evaluator.calculate_score(evaluation_results)
        
        # Store results
        key = f"{model_name}_{mode}"
        model_comparison_results[key] = results
        
        # Log metrics
        experiment.log_metrics(results.to_log())
        entities, confmatrix = results.to_confusion_matrix()
        experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
        experiment.end()
        
        # Print results
        print(f"\nResults for {model_name}:")
        print(f"  Precision: {results.pii_precision:.3f}")
        print(f"  Recall: {results.pii_recall:.3f}")
        print(f"  F-score: {results.pii_f:.3f}")
        
    except Exception as e:
        print(f"Error with model {model_name}: {str(e)}")
        continue

print("\n" + "=" * 60)
print("MODEL COMPARISON COMPLETE")
print("=" * 60)

## 8. Experiment 4: Different Entity Configurations and Thresholds

Test how different entity sets and score thresholds affect performance.

In [ ]:
# Results storage
config_comparison_results = {}

In [ ]:
%%time

print("=" * 80)
print("EXPERIMENT 4: ENTITY CONFIGURATIONS AND THRESHOLDS")
print("=" * 80)

# Create NLP engine once
nlp_engine = create_spacy_nlp_engine()

# Fixed model
model_name = GLINER_MODELS[0]
mode = "standalone"
multi_label = False

for entity_config in ENTITY_CONFIGS.keys():
    for threshold in THRESHOLD_CONFIGS:
        print(f"\n{'='*60}")
        print(f"Entity Config: {entity_config}")
        print(f"Threshold: {threshold}")
        print(f"{'='*60}\n")
        
        try:
            # Create GliNER recognizer
            gliner_recognizer = create_gliner_recognizer(
                model_name=model_name,
                gliner_labels=ENTITY_CONFIGS[entity_config],
                entity_mapping=GLINER_TO_PRESIDIO_MAPPING,
                threshold=threshold,
                multi_label=multi_label
            )
            
            # Create registry
            registry = create_registry_standalone_gliner(gliner_recognizer, nlp_engine)
            
            # Create analyzer
            analyzer = create_analyzer_engine(registry, nlp_engine, threshold=threshold)
            
            # Set up evaluator
            experiment = get_experiment_tracker()
            evaluator = SpanEvaluator(model=analyzer, iou_threshold=0.7)
            
            # Track parameters
            params = {
                "dataset_name": dataset_name,
                "model_name": evaluator.model.name,
                "gliner_model": model_name,
                "entity_config": entity_config,
                "threshold": threshold,
                "multi_label": multi_label,
                "mode": mode,
                "spacy_recognizer": "disabled"
            }
            params.update(evaluator.model.to_log())
            experiment.log_parameters(params)
            experiment.log_dataset_hash(dataset_aligned)
            
            # Run evaluation
            print("Running evaluation...")
            evaluation_results = evaluator.evaluate_all(dataset_aligned)
            results = evaluator.calculate_score(evaluation_results)
            
            # Store results
            key = f"{entity_config}_{threshold}"
            config_comparison_results[key] = results
            
            # Log metrics
            experiment.log_metrics(results.to_log())
            entities, confmatrix = results.to_confusion_matrix()
            experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
            experiment.end()
            
            # Print results
            print(f"\nResults:")
            print(f"  Precision: {results.pii_precision:.3f}")
            print(f"  Recall: {results.pii_recall:.3f}")
            print(f"  F-score: {results.pii_f:.3f}")
            
        except Exception as e:
            print(f"Error with config {entity_config}, threshold {threshold}: {str(e)}")
            continue

print("\n" + "=" * 60)
print("CONFIGURATION COMPARISON COMPLETE")
print("=" * 60)

## 9. Results Summary and Comparison

In [ ]:
def create_results_dataframe(results_dict, experiment_name):
    """Create a DataFrame from results dictionary."""
    rows = []
    for key, result in results_dict.items():
        rows.append({
            "Experiment": experiment_name,
            "Configuration": key,
            "Precision": result.pii_precision,
            "Recall": result.pii_recall,
            "F-Score": result.pii_f
        })
    return pd.DataFrame(rows)

# Combine all results
all_dfs = []
if standalone_results:
    all_dfs.append(create_results_dataframe(standalone_results, "Standalone GliNER"))
if combined_results:
    all_dfs.append(create_results_dataframe(combined_results, "GliNER + Others"))
if model_comparison_results:
    all_dfs.append(create_results_dataframe(model_comparison_results, "Model Comparison"))
if config_comparison_results:
    all_dfs.append(create_results_dataframe(config_comparison_results, "Config Comparison"))

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    print("\n" + "=" * 80)
    print("COMPLETE RESULTS SUMMARY")
    print("=" * 80)
    print(summary_df.to_string(index=False))
    
    # Save to CSV
    output_path = Path(Path.cwd().parent, "data", "gliner_evaluation_results.csv")
    summary_df.to_csv(output_path, index=False)
    print(f"\n✓ Results saved to: {output_path}")
else:
    print("No results to summarize yet. Run the experiments above.")

## 10. Visualizations

In [ ]:
import plotly.graph_objects as go

if all_dfs:
    # Create bar plot comparing all configurations
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name='Precision',
        x=summary_df['Configuration'],
        y=summary_df['Precision'],
        marker_color='lightblue'
    ))
    
    fig.add_trace(go.Bar(
        name='Recall',
        x=summary_df['Configuration'],
        y=summary_df['Recall'],
        marker_color='lightcoral'
    ))
    
    fig.add_trace(go.Bar(
        name='F-Score',
        x=summary_df['Configuration'],
        y=summary_df['F-Score'],
        marker_color='lightgreen'
    ))
    
    fig.update_layout(
        title='GliNER Performance Comparison',
        xaxis_title='Configuration',
        yaxis_title='Score',
        barmode='group',
        height=600
    )
    
    fig.show()
else:
    print("No results to visualize yet. Run the experiments above.")

## 11. Error Analysis (For Latest Experiment)

Analyze errors from the most recent experiment.

In [ ]:
# Get results from the last run
if 'results' in locals():
    print("Most Common False Positive Tokens:")
    print(ModelError.most_common_fp_tokens(results.model_errors, n=20))
    
    print("\nMost Common False Negative Tokens:")
    print(ModelError.most_common_fn_tokens(results.model_errors, n=20))
else:
    print("Run an experiment first to see error analysis.")

In [ ]:
# Detailed error analysis
if 'results' in locals():
    # False positives
    fps_df = ModelError.get_fps_dataframe(results.model_errors)
    print("Sample False Positives:")
    print(fps_df[["full_text", "token", "annotation", "prediction"]].head(10))
    
    # False negatives
    fns_df = ModelError.get_fns_dataframe(results.model_errors)
    print("\nSample False Negatives:")
    print(fns_df[["full_text", "token", "annotation", "prediction"]].head(10))
else:
    print("Run an experiment first to see detailed error analysis.")

## 12. Conclusions and Recommendations

Based on the experiments above, you can determine:

1. **Best Standalone GliNER Configuration**: Which model and entity set works best alone?
2. **Value of Combined Approach**: Does adding pattern-based recognizers improve results?
3. **Optimal Model**: Which GliNER base model provides the best trade-off between accuracy and speed?
4. **Threshold Sensitivity**: How sensitive is the model to the confidence threshold?
5. **Entity Coverage**: Which entity configuration provides the best coverage for your use case?

### Next Steps:
- Fine-tune GliNER on your specific domain data
- Experiment with custom entity labels
- Combine with domain-specific pattern recognizers
- Test on production-like data
- Monitor performance on edge cases